# BERTopic Training — 2x T4 GPU (Kaggle)

**Project:** DNLPA — Vietnamese Tech Trend Radar  
**Owner:** Member 3 — ML Engineer  
**Output chuẩn theo:** `docs/data_flow_schema_evolution.md`

### Input (đã upload lên Kaggle Dataset `data-training-bertopic`):

```
/kaggle/input/datasets/anhqunhong/data-training-bertopic/
└── training_data.csv      ← 162K rows, cột: post_id | clean_text | source
```

> `training_data.csv` đã qua toàn bộ pipeline xử lý (slang normalize, clean, dedup, truncate).  
> Notebook này chỉ **load → encode → train → evaluate → export** — không xử lý lại data.

### Output ra `/kaggle/working/bertopic_output/`:
```
embeddings.npy
post_topic_assignment.parquet
topics.parquet
bertopic_model/
  bertopic_model  (pickle)
  config.pkl
  topics.pkl
```

## Cell 1 — Kiểm tra GPU

In [1]:
import subprocess, torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  GPU {i}: {name} — {mem:.1f} GB VRAM")

assert torch.cuda.device_count() >= 1, "Cần ít nhất 1 GPU — bật GPU Accelerator trong Settings"

PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
  GPU 0: Tesla T4 — 14.6 GB VRAM
  GPU 1: Tesla T4 — 14.6 GB VRAM


## Cell 2 — Cài packages

In [2]:
%%capture
# Không cài hdbscan package → dùng sklearn.cluster.HDBSCAN (numpy 2.x native, có sẵn trên Kaggle)
# pyarrow>=15.0.0 là version đầu tiên support numpy 2.x (14.x chỉ support numpy 1.x)
# Không pin numpy → dùng Kaggle pre-installed numpy 2.0.2
!pip install -q \
    "bertopic==0.16.4" \
    "sentence-transformers>=2.7.0" \
    "umap-learn>=0.5.7" \
    "gensim>=4.3.3" \
    "pyarrow>=15.0.0"

## Cell 3 — Import & Config

In [3]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GRPC_VERBOSITY"] = "ERROR"
os.environ["GLOG_minloglevel"] = "3"

import json, os, re, time, pickle, sys, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

import torch
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

warnings.filterwarnings('ignore')

print(f"numpy  : {np.__version__}")
import sklearn; print(f"sklearn: {sklearn.__version__}")

# ── Input: training_data.csv đã xử lý sẵn ──
_DATASET_DIR = "/kaggle/input/datasets/anhqunhong/data-training-bertopic"
PROCESSED_CSV = Path(f"{_DATASET_DIR}/training_data.csv")

# ── Output ──
OUTPUT_DIR = Path("/kaggle/working/bertopic_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Model config ──
EMBEDDING_MODEL = "vinai/phobert-base"
MODEL_VERSION   = "bertopic_v1"

# ── BERTopic hyperparams ──
N_NEIGHBORS      = 15
N_COMPONENTS     = 5
MIN_DIST         = 0.0
MIN_CLUSTER_SIZE = 15
MIN_SAMPLES      = 10
TOP_N_WORDS      = 10
BATCH_SIZE       = 128   # 256→128: giảm để tránh OOM/freeze trên T4 14GB

print("Config loaded.")
print(f"Input CSV  : {PROCESSED_CSV}")
print(f"Output dir : {OUTPUT_DIR}")

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.
E0000 00:00:1779244971.563010      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779244971.664728      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779244972.630736      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779244972.630776      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779244972.630779      23 computation_placer.cc:177] computation placer already registered. Please check link

numpy  : 2.0.2
sklearn: 1.6.1
Config loaded.
Input CSV  : /kaggle/input/datasets/anhqunhong/data-training-bertopic/training_data.csv
Output dir : /kaggle/working/bertopic_output


## Cell 4 — Load `training_data.csv` (đã xử lý sẵn)

`training_data.csv` trong Kaggle Dataset đã qua pipeline đầy đủ → load thẳng, không xử lý lại.

In [4]:
print(f"Loading {PROCESSED_CSV} ...")
df = pd.read_csv(PROCESSED_CSV)
print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}")

assert 'clean_text' in df.columns and 'post_id' in df.columns, \
    "CSV thiếu cột — cần 'post_id' và 'clean_text'"

df = df.dropna(subset=['clean_text'])
df = df[df['clean_text'].str.len() >= 10]
df['post_id'] = df['post_id'].astype(str)

print(f"Rows sau filter : {len(df):,}")
if 'source' in df.columns:
    print(f"Source breakdown: {df['source'].value_counts().to_dict()}")
print(f"Text len (mean) : {df['clean_text'].str.len().mean():.0f} chars")
print(f"\nSample: {df['clean_text'].iloc[0][:120]}...")

documents = df['clean_text'].tolist()
post_ids  = df['post_id'].tolist()
print(f"\n✅ {len(documents):,} documents sẵn sàng để encode")

Loading /kaggle/input/datasets/anhqunhong/data-training-bertopic/training_data.csv ...
Shape   : (162691, 3)
Columns : ['post_id', 'clean_text', 'source']
Rows sau filter : 162,691
Source breakdown: {'voz': 138537, 'vnexpress': 23027, 'vatvo': 1127}
Text len (mean) : 143 chars

Sample: copilot trên windows 11 sắp tích hợp vào taskbar, thay thế windows search microsoft vừa công_bố một bản cập_nhật quan_tr...

✅ 162,691 documents sẵn sàng để encode


## Cell 5 — Encode với PhoBERT (2x T4, sequential)

**Tại sao encode riêng trước BERTopic?**
- Encode ~80% thời gian — tách ra để có thể re-run UMAP/HDBSCAN mà không encode lại
- Dùng **sequential encoding** (không dùng `encode_multi_process`) — tránh deadlock trên T4 khi văn bản dài

> Nếu restart kernel và `embeddings.npy` đã có → chạy cell cache-load bên dưới để bỏ qua encode.

In [5]:
print(f"Loading {EMBEDDING_MODEL}...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

# PhoBERT max 256 tokens — set tường minh để tránh OOM silent error
embedding_model.max_seq_length = 256

n_gpus = torch.cuda.device_count()
print(f"GPUs available : {n_gpus}")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Max seq length : {embedding_model.max_seq_length}")

t0 = time.time()

# Không dùng encode_multi_process — hay bị deadlock khi văn bản dài trên T4
# Chia đôi corpus, encode tuần tự trên từng GPU → ổn định hơn
if n_gpus >= 2:
    mid = len(documents) // 2
    docs_0, docs_1 = documents[:mid], documents[mid:]
    print(f"\nSplit: GPU 0 ← {len(docs_0):,} | GPU 1 ← {len(docs_1):,}")

    torch.cuda.empty_cache()
    print("Encoding GPU 0...")
    emb_0 = embedding_model.encode(
        docs_0, batch_size=BATCH_SIZE, show_progress_bar=True,
        device='cuda:0', convert_to_numpy=True,
    )
    torch.cuda.empty_cache()

    print("Encoding GPU 1...")
    emb_1 = embedding_model.encode(
        docs_1, batch_size=BATCH_SIZE, show_progress_bar=True,
        device='cuda:1', convert_to_numpy=True,
    )
    torch.cuda.empty_cache()
    embeddings = np.vstack([emb_0, emb_1])

elif n_gpus == 1:
    torch.cuda.empty_cache()
    print(f"Encoding {len(documents):,} docs trên GPU 0...")
    embeddings = embedding_model.encode(
        documents, batch_size=BATCH_SIZE, show_progress_bar=True,
        device='cuda:0', convert_to_numpy=True,
    )
else:
    print("⚠️  CPU mode — chậm (~30-60 phút với 162K docs)")
    embeddings = embedding_model.encode(
        documents, batch_size=32, show_progress_bar=True, convert_to_numpy=True,
    )

elapsed = time.time() - t0
print(f"\nEncoding xong: {len(documents):,} docs trong {elapsed:.1f}s")
print(f"Embeddings shape: {embeddings.shape}")

np.save(OUTPUT_DIR / "embeddings.npy", embeddings)
print(f"Saved → {OUTPUT_DIR}/embeddings.npy")

Loading vinai/phobert-base...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

GPUs available : 2
Batch size     : 128
Max seq length : 256

Split: GPU 0 ← 81,345 | GPU 1 ← 81,346
Encoding GPU 0...


Batches:   0%|          | 0/636 [00:00<?, ?it/s]

Encoding GPU 1...


Batches:   0%|          | 0/636 [00:00<?, ?it/s]


Encoding xong: 162,691 docs trong 503.8s
Embeddings shape: (162691, 768)
Saved → /kaggle/working/bertopic_output/embeddings.npy


## Cell 6 — Build & Train BERTopic

In [6]:
umap_model = UMAP(
    n_neighbors=N_NEIGHBORS,
    n_components=N_COMPONENTS,
    min_dist=MIN_DIST,
    metric='cosine',
    random_state=42,
    low_memory=False,
)

# sklearn HDBSCAN (không cần hdbscan package, tránh numpy 2.x conflict)
hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    metric='euclidean',
    cluster_selection_method='eom',
    n_jobs=-1,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=TOP_N_WORDS,
    calculate_probabilities=False,
    verbose=True,
)

print("Training BERTopic...")
t0 = time.time()
topics, probs = topic_model.fit_transform(documents, embeddings=embeddings)
elapsed = time.time() - t0

n_topics   = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers = sum(t == -1 for t in topics)

print(f"\n{'='*50}")
print(f"TRAINING XONG trong {elapsed:.1f}s")
print(f"Topics tìm được : {n_topics}")
print(f"Outliers (-1)   : {n_outliers} ({n_outliers/len(topics)*100:.1f}%)")
print(f"Docs có topic   : {len(topics) - n_outliers}")
print(f"{'='*50}")

2026-05-20 02:53:30,386 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training BERTopic...


2026-05-20 02:58:26,117 - BERTopic - Dimensionality - Completed ✓
2026-05-20 02:58:26,123 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-20 03:01:45,632 - BERTopic - Cluster - Completed ✓
2026-05-20 03:01:45,673 - BERTopic - Representation - Extracting topics from clusters using representation models.
2026-05-20 03:01:49,690 - BERTopic - Representation - Completed ✓



TRAINING XONG trong 500.4s
Topics tìm được : 214
Outliers (-1)   : 111023 (68.2%)
Docs có topic   : 51668


## Cell 7 — Xem Topics

In [7]:
print("Top words per topic:")
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    words = topic_model.get_topic(tid)
    top3  = [w for w, _ in words[:3]]
    count = topics.count(tid)
    print(f"  Topic {tid:2d} ({count:5d} docs): {' | '.join(top3)}")

Top words per topic:
  Topic  0 (17925 docs): fen | thím | đc
  Topic  1 ( 3597 docs): xe | km | đi
  Topic  2 ( 3594 docs): cài | app | win
  Topic  3 ( 3522 docs): thenextvoz | for | via
  Topic  4 ( 3169 docs): sale | giá | hàng
  Topic  5 ( 2233 docs): nan | tại | chính_thức
  Topic  6 ( 1939 docs): phòng | nhiệt_độ | nước
  Topic  7 ( 1303 docs): hê | trung_quốc | huawei
  Topic  8 ( 1179 docs): sạc | cắm | pin
  Topic  9 ( 1068 docs): con_người | ai | sẽ
  Topic 10 (  806 docs): đánh | đánh_giá | giá
  Topic 11 (  778 docs): chụp | ảnh | mm
  Topic 12 (  537 docs): phim | xem | nhân_vật
  Topic 13 (  438 docs): vozfapp | gửi | bằng
  Topic 14 (  385 docs): loa | nghe | bass
  Topic 15 (  382 docs): the | com | with
  Topic 16 (  315 docs): mày | tao | chửi
  Topic 17 (  275 docs): quán | ăn | mì
  Topic 18 (  261 docs): vậy | bao_nhiêu | đâu
  Topic 19 (  243 docs): thi | cung | vơi
  Topic 20 (  222 docs): xin | link | địa_chỉ
  Topic 21 (  212 docs): attachments | views | kb
  

## Cell 8 — Benchmark Evaluation

| Metric | Ý nghĩa | Tốt khi |
|---|---|---|
| **Coherence C_V** | Ngữ nghĩa topic, so được với LDA baseline | > 0.50 |
| **Coherence NPMI** | Nhạy hơn C_V với corpus nhỏ | > −0.10 |
| **Topic Diversity** | % unique keywords across all topics | > 0.70 |
| **Silhouette (UMAP5D)** | Chất lượng cluster trong không gian 5D | > 0.30 |
| **Outlier %** | Docs không được gán topic | < 30% |

In [8]:
print("=" * 55)
print("BENCHMARK EVALUATION")
print("=" * 55)

topic_words = []
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(tid)]
    topic_words.append(words)

texts      = [doc.split() for doc in documents]
dictionary = Dictionary(texts)

# ── 1. Coherence C_V ──
print("\n[1/4] Coherence C_V (baseline LDA: 0.5736)...")
coherence_cv = CoherenceModel(
    topics=topic_words, texts=texts, dictionary=dictionary,
    coherence='c_v', processes=1,
).get_coherence()
print(f"   C_V    : {coherence_cv:.4f}")

# ── 2. Coherence NPMI ──
print("\n[2/4] Coherence NPMI...")
coherence_npmi = CoherenceModel(
    topics=topic_words, texts=texts, dictionary=dictionary,
    coherence='c_npmi', processes=1,
).get_coherence()
print(f"   NPMI   : {coherence_npmi:.4f}  (tốt nếu > -0.1)")

# ── 3. Topic Diversity ──
print("\n[3/4] Topic Diversity...")
all_top_words = [w for words in topic_words for w in words[:10]]
diversity = len(set(all_top_words)) / len(all_top_words) if all_top_words else 0.0
print(f"   Diversity: {diversity:.4f}  (tốt nếu > 0.7)")

# ── 4. Silhouette Score trên UMAP 5D ──
print("\n[4/4] Silhouette Score (UMAP 5D, sample 10K)...")
non_outlier_idx = np.where(np.array(topics) != -1)[0]
sil = None
if len(non_outlier_idx) > 2:
    sample_idx = non_outlier_idx
    if len(sample_idx) > 10000:
        rng = np.random.default_rng(42)
        sample_idx = rng.choice(non_outlier_idx, 10000, replace=False)
    umap_embed = topic_model.umap_model.transform(embeddings[sample_idx])
    sil = silhouette_score(
        umap_embed, np.array(topics)[sample_idx],
        sample_size=min(5000, len(sample_idx)), random_state=42,
    )
    print(f"   Silhouette: {sil:.4f}  (tốt nếu > 0.3)")
else:
    print("   Silhouette: N/A")

# ── Summary table ──
print(f"\n{'='*55}")
print("SUMMARY SO SÁNH VỚI LDA BASELINE")
print(f"{'='*55}")
print(f"  {'Metric':<22} {'BERTopic':>10}  {'LDA':>10}")
print(f"  {'-'*44}")
print(f"  {'Coherence C_V':<22} {coherence_cv:>10.4f}  {'0.5736':>10}")
print(f"  {'Coherence NPMI':<22} {coherence_npmi:>10.4f}  {'N/A':>10}")
print(f"  {'Topic Diversity':<22} {diversity:>10.4f}  {'N/A':>10}")
if sil is not None:
    print(f"  {'Silhouette (UMAP5D)':<22} {sil:>10.4f}  {'N/A':>10}")
print(f"  {'Outlier %':<22} {n_outliers/len(topics)*100:>9.1f}%  {'N/A':>10}")
print(f"  {'N Topics':<22} {n_topics:>10}  {'N/A':>10}")

verdict = "✅ Tốt" if coherence_cv > 0.5 else ("⚠️  Chấp nhận" if coherence_cv > 0.35 else "❌ Kém")
print(f"\n  C_V vs LDA: {verdict}")

BENCHMARK EVALUATION

[1/4] Coherence C_V (baseline LDA: 0.5736)...
   C_V    : 0.5368

[2/4] Coherence NPMI...
   NPMI   : 0.0505  (tốt nếu > -0.1)

[3/4] Topic Diversity...
   Diversity: 0.7248  (tốt nếu > 0.7)

[4/4] Silhouette Score (UMAP 5D, sample 10K)...
   Silhouette: 0.0758  (tốt nếu > 0.3)

SUMMARY SO SÁNH VỚI LDA BASELINE
  Metric                   BERTopic         LDA
  --------------------------------------------
  Coherence C_V              0.5368      0.5736
  Coherence NPMI             0.0505         N/A
  Topic Diversity            0.7248         N/A
  Silhouette (UMAP5D)        0.0758         N/A
  Outlier %                   68.2%         N/A
  N Topics                      214         N/A

  C_V vs LDA: ✅ Tốt


## Cell 9 — Export kết quả (`stg_post_topics` + `stg_topics`)

In [9]:
now = datetime.now(tz=timezone.utc)

# ── post_topic_assignment.parquet → stg_post_topics ──
if probs is not None and hasattr(probs, 'shape') and len(probs.shape) == 2:
    topic_probs = probs.max(axis=1).tolist()
else:
    topic_probs = [1.0] * len(topics)

post_topics_df = pd.DataFrame({
    'post_id'          : post_ids,
    'topic_id'         : [int(t) for t in topics],
    'topic_probability': [float(p) for p in topic_probs],
    'model_type'       : 'bertopic',
    'predicted_at'     : now,
})
post_topics_df.to_parquet(OUTPUT_DIR / 'post_topic_assignment.parquet', index=False)
print(f"✅ post_topic_assignment.parquet — {len(post_topics_df):,} rows")

# ── topics.parquet → stg_topics ──
topics_rows = []
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(tid)]
    topics_rows.append({
        'topic_id'       : int(tid),
        'label'          : '_'.join(words[:3]),
        'top_keywords'   : words,
        'coherence_score': float(coherence_cv),
        'model_version'  : MODEL_VERSION,
        'created_at'     : now,
    })
topics_df = pd.DataFrame(topics_rows)
topics_df.to_parquet(OUTPUT_DIR / 'topics.parquet', index=False)
print(f"✅ topics.parquet — {len(topics_df)} topics")

display(topics_df[['topic_id', 'label', 'coherence_score', 'model_version']])

✅ post_topic_assignment.parquet — 162,691 rows
✅ topics.parquet — 214 topics


,topic_id,label,coherence_score,model_version
0,0,fen_thím_đc,0.536843,bertopic_v1
1,1,xe_km_đi,0.536843,bertopic_v1
2,2,cài_app_win,0.536843,bertopic_v1
3,3,thenextvoz_for_via,0.536843,bertopic_v1
4,4,sale_giá_hàng,0.536843,bertopic_v1
...,...,...,...,...
209,209,poe_dự_đc_mỹ_đế,0.536843,bertopic_v1
210,210,phev_hev_hybrid,0.536843,bertopic_v1
211,211,chuẩn_nha_hn_ngon_lun,0.536843,bertopic_v1
212,212,trưng_showroom_tội_thọt_qua_động,0.536843,bertopic_v1


## Cell 10 — Save Model Artifacts

In [10]:
model_dir = OUTPUT_DIR / 'bertopic_model'
model_dir.mkdir(exist_ok=True)

topic_model.save(
    str(model_dir / 'bertopic_model'),
    serialization='pickle',
    save_embedding_model=False,
)
print(f"✅ BERTopic model → {model_dir}/bertopic_model")

config = {
    'embedding_model'  : EMBEDDING_MODEL,
    'model_version'    : MODEL_VERSION,
    'n_neighbors'      : N_NEIGHBORS,
    'n_components'     : N_COMPONENTS,
    'min_dist'         : MIN_DIST,
    'min_cluster_size' : MIN_CLUSTER_SIZE,
    'min_samples'      : MIN_SAMPLES,
    'top_n_words'      : TOP_N_WORDS,
    'trained_on'       : now.isoformat(),
    'n_documents'      : len(documents),
    'n_topics'         : n_topics,
    'coherence_cv'     : float(coherence_cv),
    'coherence_npmi'   : float(coherence_npmi),
    'topic_diversity'  : float(diversity),
    'silhouette_score' : float(sil) if sil is not None else None,
    'outlier_pct'      : round(n_outliers / len(topics) * 100, 2),
}
with open(model_dir / 'config.pkl', 'wb') as f:
    pickle.dump(config, f)
print(f"✅ Config → {model_dir}/config.pkl")

with open(model_dir / 'topics.pkl', 'wb') as f:
    pickle.dump({'topics': topics, 'probs': probs}, f)
print(f"✅ Topics/probs → {model_dir}/topics.pkl")

# ── Output summary ──
print(f"\n{'='*55}")
print("OUTPUT /kaggle/working/ (tải về máy):")
print(f"{'='*55}")
for f in sorted(Path('/kaggle/working').rglob('*')):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f"  {str(f.relative_to('/kaggle/working')):50s} {size:8.1f} KB")

print(f"\n{'='*55}")
print("FINAL SUMMARY")
print(f"{'='*55}")
print(f"  Documents  : {len(documents):,}")
print(f"  Topics     : {n_topics}")
print(f"  Outliers   : {n_outliers} ({n_outliers/len(topics)*100:.1f}%)")
print(f"  C_V        : {coherence_cv:.4f}")
print(f"  NPMI       : {coherence_npmi:.4f}")
print(f"  Diversity  : {diversity:.4f}")
if sil is not None:
    print(f"  Silhouette : {sil:.4f}")
print(f"  Model      : {MODEL_VERSION} / {EMBEDDING_MODEL}")

2026-05-20 03:04:15,413 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✅ BERTopic model → /kaggle/working/bertopic_output/bertopic_model/bertopic_model
✅ Config → /kaggle/working/bertopic_output/bertopic_model/config.pkl
✅ Topics/probs → /kaggle/working/bertopic_output/bertopic_model/topics.pkl

OUTPUT /kaggle/working/ (tải về máy):
  __notebook__.ipynb                                     59.4 KB
  bertopic_output/bertopic_model/bertopic_model      1737245.5 KB
  bertopic_output/bertopic_model/config.pkl               0.4 KB
  bertopic_output/bertopic_model/topics.pkl            1914.6 KB
  bertopic_output/embeddings.npy                     488073.1 KB
  bertopic_output/post_topic_assignment.parquet        2258.6 KB
  bertopic_output/topics.parquet                         23.8 KB

FINAL SUMMARY
  Documents  : 162,691
  Topics     : 214
  Outliers   : 111023 (68.2%)
  C_V        : 0.5368
  NPMI       : 0.0505
  Diversity  : 0.7248
  Silhouette : 0.0758
  Model      : bertopic_v1 / vinai/phobert-base


## Cell 12 — Sau khi train: đẩy về local / HDFS

Tải toàn bộ `/kaggle/working/` về máy, sau đó:

```bash
# Copy model vào project local để test
cp -r bertopic_output/bertopic_model/ \
    output/task3.1_bertopic/output/bertopic_model/

# Đẩy lên HDFS (khi deploy cluster)
hdfs dfs -mkdir -p /data/models/bertopic/
hdfs dfs -put -f bertopic_output/bertopic_model/ /data/models/bertopic/
hdfs dfs -put -f bertopic_output/post_topic_assignment.parquet \
    /data/silver/post_topics/model=bertopic/
hdfs dfs -put -f bertopic_output/topics.parquet /data/silver/topics/
```